# Gemini 01. 기간별 API 사용량 조회하기

## 먼저 알아둘 것: API Key로 사용량을 조회하는 API가 없다

Claude는 Admin Key가 있으면 Usage & Cost API로 기간별 사용량을 조회할 수 있었다(Claude 04 노트북).
Gemini API(Google AI Studio 키)에는 **기간별 사용량이나 비용을 조회하는 공개 API가 없다.** (2026-10 공식 문서 기준)

그래서 사용량은 다음 두 가지로 확인한다.

1. **웹 화면에서 확인** (공식, 전체 사용량)
2. **요청할 때마다 직접 기록** (코드로 조회 가능, 이 코드로 보낸 요청만)

## 1. 웹 화면에서 확인하는 곳

| 확인할 것 | 위치 |
|---|---|
| 사용량 (요청 수, 토큰) | AI Studio → Dashboard → [Usage](https://aistudio.google.com/usage) |
| 현재 Rate Limit과 사용률 | AI Studio → [Rate limit](https://aistudio.google.com/rate-limit) |
| 결제 잔액, 결제 내역 (선불) | AI Studio → [Billing](https://aistudio.google.com/billing) |
| 프로젝트 월 지출 한도 | AI Studio → [Spend](https://aistudio.google.com/spend) |
| 요금 등급 (Free / Tier 1~3) | AI Studio → [Projects](https://aistudio.google.com/projects) |
| 기간별 비용 리포트 | Google Cloud 콘솔 → [Billing reports](https://console.cloud.google.com/billing/reports) (Services를 Gemini API로 필터, 최대 24시간 지연) |

> **코드로 비용 데이터를 꼭 받아야 한다면** Google Cloud의 결제 데이터를 BigQuery로 내보내는(Billing export) 방법이 있다. Google Cloud 프로젝트 설정과 별도 인증(서비스 계정 등)이 필요하므로 이 노트북에서는 다루지 않는다.

### Claude와 다른 점

- **Rate Limit은 API Key가 아니라 프로젝트 단위**로 적용된다. 같은 프로젝트의 키를 여러 개 써도 한도를 나눠 쓴다.
- **일일 요청 수(RPD) 한도는 미국 태평양 시간 자정에 초기화**된다. 한국 시간으로는 오후 4시(서머타임 기간) 또는 오후 5시다.
- 무료 등급(Free tier)이 있다. 무료 등급에서는 비용이 0이지만 Rate Limit이 낮다.

## 2. 준비

- 패키지: `google-genai` (Google 공식 SDK)
- `genai.Client()`는 환경변수 `GEMINI_API_KEY`를 자동으로 읽는다. (`GOOGLE_API_KEY`도 함께 있으면 `GOOGLE_API_KEY`가 우선)
- `load_dotenv()`는 현재 폴더에서 상위 폴더로 올라가며 `.env`를 찾으므로, `gemini/` 폴더 안에서도 프로젝트 루트의 `.env`를 읽는다.

In [1]:
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

MODEL = "gemini-3.8-flash"

## 3. 응답 1회의 사용량: `usage_metadata`

Claude의 `response.usage`에 해당한다. 필드 이름과 구성이 다르다.

| Gemini 필드 | 의미 | Claude에서 대응하는 것 |
|---|---|---|
| `prompt_token_count` | 입력 토큰 (캐시된 부분 **포함**) | `input_tokens` + 캐시 관련 입력 |
| `cached_content_token_count` | 입력 중 캐시에서 읽은 토큰 | `cache_read_input_tokens` |
| `candidates_token_count` | 답변 토큰 (thinking **제외**) | `output_tokens` − thinking |
| `thoughts_token_count` | thinking 토큰 | `output_tokens_details.thinking_tokens` |
| `total_token_count` | 전체 합계 | (없음, 직접 계산) |

**주의: 출력 요금은 `candidates_token_count + thoughts_token_count`에 매겨진다.** Claude는 `output_tokens`에 thinking이 이미 포함되어 있었지만, Gemini는 따로 나와 있어서 더해야 한다.

> 아래 셀은 요청을 1회 보내므로 소액의 요금이 발생한다. (무료 등급이면 무료)

In [2]:
response = client.models.generate_content(
    model=MODEL,
    contents="토큰이 뭔지 한 문장으로 설명해줘.",
)
print(response.text)

u = response.usage_metadata
print(f"\n입력      : {u.prompt_token_count or 0:>6,}")
print(f"  캐시    : {u.cached_content_token_count or 0:>6,}")
print(f"답변      : {u.candidates_token_count or 0:>6,}")
print(f"thinking  : {u.thoughts_token_count or 0:>6,}")
print(f"합계      : {u.total_token_count or 0:>6,}")
print(f"\n응답 모델 : {response.model_version}")

**토큰이란 인공지능이 텍스트를 인식하고 처리하기 위해 문장을 잘게 쪼갠 가장 작은 언어 단위입니다.**

입력      :     14
  캐시    :      0
답변      :     32
thinking  :    583
합계      :    629

응답 모델 : gemini-3.8-flash


> 짧은 질문인데도 thinking 토큰이 답변보다 훨씬 많을 수 있다. 테스트에서는 답변 34토큰에 thinking 589토큰이 나왔다. Gemini 3.x Flash는 기본적으로 thinking을 많이 쓰는 편이므로, 비용을 볼 때 thinking을 꼭 함께 봐야 한다.

## 4. 요청할 때마다 직접 기록하기

Claude 04 노트북의 5번과 같은 방식이다. `usage_metadata`를 CSV 파일에 한 줄씩 쌓고, 나중에 기간별로 합산한다.

- 기록 파일: `gemini/log/gemini_usage_log.csv` (폴더가 없으면 자동 생성)
- 한계: **이 코드로 보낸 요청만** 기록된다. AI Studio 화면이나 다른 프로그램에서 쓴 양은 빠진다.

In [3]:
import csv
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

LOG_DIR = Path("log")  # 노트북 위치 기준 → gemini/log
LOG_FILE = LOG_DIR / "gemini_usage_log.csv"
FIELDS = ["timestamp_utc", "model", "prompt_tokens", "cached_tokens", "answer_tokens", "thinking_tokens", "total_tokens"]


def log_usage(response) -> None:
    """응답의 usage_metadata를 CSV에 한 줄 추가"""
    u = response.usage_metadata
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "model": response.model_version,  # 실제로 응답한 모델 버전
        "prompt_tokens": u.prompt_token_count or 0,  # None일 수 있으므로 0으로
        "cached_tokens": u.cached_content_token_count or 0,
        "answer_tokens": u.candidates_token_count or 0,
        "thinking_tokens": u.thoughts_token_count or 0,
        "total_tokens": u.total_token_count or 0,
    }
    LOG_DIR.mkdir(exist_ok=True)
    is_new = not LOG_FILE.exists()
    with LOG_FILE.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow(row)


# 요청할 때마다 log_usage(response)를 함께 호출
log_usage(response)  # 위 3번에서 받은 응답
print(f"기록 완료 → {LOG_FILE}")

기록 완료 → log/gemini_usage_log.csv


## 5. 기록에서 기간별로 합산하기

In [4]:
def read_log(start: datetime, end: datetime) -> list[dict]:
    """기록 파일에서 [start, end) 기간의 행만 반환"""
    if not LOG_FILE.exists():
        print(f"{LOG_FILE} 파일이 없습니다. 먼저 log_usage()로 기록하세요.")
        return []
    rows = []
    with LOG_FILE.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ts = datetime.strptime(row["timestamp_utc"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if start <= ts < end:
                row["ts"] = ts
                rows.append(row)
    return rows


def summarize(rows: list[dict]) -> None:
    totals = defaultdict(lambda: defaultdict(int))
    for row in rows:
        t = totals[row["model"]]
        t["requests"] += 1
        for key in ("prompt_tokens", "answer_tokens", "thinking_tokens", "total_tokens"):
            t[key] += int(row[key])

    print(f"{'Model':<22} {'Requests':>8} {'Input':>9} {'Answer':>9} {'Thinking':>9} {'Total':>9}")
    print("-" * 71)
    for model, t in sorted(totals.items()):
        print(f"{model:<22} {t['requests']:>8} {t['prompt_tokens']:>9,} {t['answer_tokens']:>9,} "
              f"{t['thinking_tokens']:>9,} {t['total_tokens']:>9,}")


# 최근 7일
now = datetime.now(timezone.utc)
summarize(read_log(now - timedelta(days=7), now))

Model                  Requests     Input    Answer  Thinking     Total
-----------------------------------------------------------------------
gemini-3.8-flash              1        14        32       583       629


### 오늘 요청 수 확인 (RPD 기준)

RPD(일일 요청 수) 한도는 **미국 태평양 시간 자정**에 초기화되므로, "오늘"도 태평양 시간 기준으로 잘라야 한도와 비교할 수 있다.
한도 값은 [AI Studio의 Rate limit 페이지](https://aistudio.google.com/rate-limit)에서 확인한다.

In [5]:
PACIFIC = ZoneInfo("America/Los_Angeles")
KST = ZoneInfo("Asia/Seoul")

today_start = datetime.now(PACIFIC).replace(hour=0, minute=0, second=0, microsecond=0)
next_reset = today_start + timedelta(days=1)

rows_today = read_log(today_start.astimezone(timezone.utc), datetime.now(timezone.utc))
print(f"오늘(태평양 시간) 기록된 요청 수: {len(rows_today)}건")
print(f"다음 초기화 시각: {next_reset.astimezone(KST):%Y-%m-%d %H:%M} (한국 시간)")

오늘(태평양 시간) 기록된 요청 수: 1건
다음 초기화 시각: 2026-10-20 16:00 (한국 시간)


## 정리

| 하고 싶은 것 | Claude | Gemini |
|---|---|---|
| 요청 1회 사용량 | `response.usage` | `response.usage_metadata` |
| 출력 토큰 | `output_tokens` (thinking 포함) | `candidates_token_count` + `thoughts_token_count` |
| 기간별 사용량 (API) | Usage API (Admin Key 필요) | **공개 API 없음** |
| 기간별 사용량 (화면) | Console Usage 페이지 | AI Studio Usage 페이지 |
| 기간별 비용 (화면) | Console Cost 페이지 | Google Cloud Billing reports |
| 직접 기록 | `log/usage_log.csv` | `gemini/log/gemini_usage_log.csv` |